In [85]:
import pandas as pd
import pandas_ta as ta
import lightgbm as lgb
import numpy as np

## Loading the Data

In [2]:
df = pd.read_csv('btc_1hour_cleaned.csv')

In [3]:
df.head(2)

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0


## Setting the correct formats and cleaning Dataframe

In [4]:
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

In [5]:
df = df.set_index('Open time')

In [6]:
df.head(3)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
Open time,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,0


In [7]:
df.shape

(71588, 11)

# Feature Engineering

## Functions for features: Strategy 1 + regime

In [8]:
# ROC
def compute_roc(df):
    df['roc_10'] = df['Close'].pct_change(periods=10)
    df['roc_21'] = df['Close'].pct_change(periods=21)
    return df

# MACD Histogram
def compute_macd(df):
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd_histogram'] = macd['MACDh_12_26_9']
    return df

# ADX - Used as a feature and for regime detection
def compute_adx(df):
    df['adx'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    return df

# SMA 50 and SMA 200 is purely used for regime detection but not in feature matrix
def compute_regime(df):
    df['sma_50']  = df['Close'].rolling(50).mean()
    df['sma_200'] = df['Close'].rolling(200).mean()
    df['regime']  = (df['sma_50'] > df['sma_200']).astype(int).replace(0, -1)
    return df

# ATR 14
def compute_atr(df):
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    return df

# NATR
def compute_natr(df):
    df['natr'] = ta.natr(df['High'], df['Low'], df['Close'], length=14)
    return df

## Pipeline

In [9]:
def feature_pipeline(df):
    steps = [
        # model features (go to X)
        compute_roc,
        compute_macd,
        compute_adx,
        compute_atr,
        compute_natr,
        # execution layer only (do not go to X)
        compute_regime
    ]
    for step in steps:
        df = step(df)
    return df

In [10]:
df = feature_pipeline(df)

In [11]:
df.head(20)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore,roc_10,roc_21,macd_histogram,adx,atr_14,natr,sma_50,sma_200,regime
Open time,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,2018-01-01 03:59:59.999,5.657448e+06,4789,137.918407,1.858041e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,2018-01-01 04:59:59.999,4.588047e+06,4563,172.957635,2.328058e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 05:00:00,13615.20,13699.00,13526.50,13558.99,404.229046,2018-01-01 05:59:59.999,5.499055e+06,5086,142.331058,1.935710e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 06:00:00,13539.00,13800.00,13510.00,13780.41,264.989684,2018-01-01 06:59:59.999,3.613408e+06,4072,126.077500,1.718753e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 07:00:00,13780.00,13818.55,13555.02,13570.35,292.188777,2018-01-01 07:59:59.999,4.002026e+06,4340,147.150029,2.016275e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 08:00:00,13569.98,13735.24,13400.00,13499.99,271.813553,2018-01-01 08:59:59.999,3.681944e+06,3733,122.809696,1.664737e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1


# Creating y

In [12]:
# Lookahead window - change this value to test different horizons
N = 5

# Step 1 - Forward return
df['forward_return'] = df['Close'].pct_change(periods=N).shift(-N)

In [13]:
# Step 2 - Binary Label
df['label'] = (df['forward_return'] > 0).astype(int)
# 1 - BUY, 0 - SELL

In [14]:
df.head(10)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,...,roc_21,macd_histogram,adx,atr_14,natr,sma_50,sma_200,regime,forward_return,label
Open time,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.002216,1
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.043728,1
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.018017,1
2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,2018-01-01 03:59:59.999,5.657448e+06,4789,137.918407,1.858041e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.006708,1
2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,2018-01-01 04:59:59.999,4.588047e+06,4563,172.957635,2.328058e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.001175,1
2018-01-01 05:00:00,13615.20,13699.00,13526.50,13558.99,404.229046,2018-01-01 05:59:59.999,5.499055e+06,5086,142.331058,1.935710e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.000813,1
2018-01-01 06:00:00,13539.00,13800.00,13510.00,13780.41,264.989684,2018-01-01 06:59:59.999,3.613408e+06,4072,126.077500,1.718753e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-0.040627,0
2018-01-01 07:00:00,13780.00,13818.55,13555.02,13570.35,292.188777,2018-01-01 07:59:59.999,4.002026e+06,4340,147.150029,2.016275e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-0.029323,0
2018-01-01 08:00:00,13569.98,13735.24,13400.00,13499.99,271.813553,2018-01-01 08:59:59.999,3.681944e+06,3733,122.809696,1.664737e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-0.035777,0


In [15]:
df

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,...,roc_21,macd_histogram,adx,atr_14,natr,sma_50,sma_200,regime,forward_return,label
Open time,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.002216,1
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.043728,1
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.018017,1
2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,2018-01-01 03:59:59.999,5.657448e+06,4789,137.918407,1.858041e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.006708,1
2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,2018-01-01 04:59:59.999,4.588047e+06,4563,172.957635,2.328058e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.001175,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-07 17:00:00,67783.63,67935.01,67768.97,67788.41,217.681040,2026-03-07 17:59:59.999,1.477294e+07,41860,146.943900,9.972905e+06,...,-0.004737,86.079662,49.233226,364.396069,0.435665,69260.8266,68305.40545,1,NaN,0
2026-03-07 18:00:00,67788.40,67788.41,67480.00,67568.13,1180.247820,2026-03-07 18:59:59.999,7.979030e+07,112179,388.463060,2.626513e+07,...,-0.010558,67.496503,49.596516,360.397064,0.439667,69186.0708,68310.22155,1,NaN,0
2026-03-07 19:00:00,67568.13,67572.35,66915.26,67080.34,1312.230200,2026-03-07 19:59:59.999,8.822703e+07,161182,521.428370,3.504474e+07,...,-0.018143,25.262143,50.705968,381.589417,0.514423,69113.5894,68314.98830,1,NaN,0


In [16]:
df.shape

(71588, 22)

In [17]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore', 'roc_10', 'roc_21',
       'macd_histogram', 'adx', 'atr_14', 'natr', 'sma_50', 'sma_200',
       'regime', 'forward_return', 'label'],
      dtype='str')

## Creating the lags

In [18]:
# Number of lags to create
N_lags = 5

feature_cols = ['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx']

# Create lags for all feature columns
for col in feature_cols:
    for lag in range(1, N_lags + 1):
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Drop NaN rows created by the lags
df = df.dropna(subset=feature_cols + [f'{col}_lag{i}' for col in feature_cols for i in range(1, N_lags + 1)] + ['label'])

In [19]:
df.shape

(71550, 47)

In [20]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore', 'roc_10', 'roc_21',
       'macd_histogram', 'adx', 'atr_14', 'natr', 'sma_50', 'sma_200',
       'regime', 'forward_return', 'label', 'Volume_lag1', 'Volume_lag2',
       'Volume_lag3', 'Volume_lag4', 'Volume_lag5', 'roc_10_lag1',
       'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
       'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4',
       'roc_21_lag5', 'macd_histogram_lag1', 'macd_histogram_lag2',
       'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
       'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5'],
      dtype='str')

## Creating the Feature Matrix and y

In [21]:
X = df[['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx',
                     'Volume_lag1', 'Volume_lag2', 'Volume_lag3', 'Volume_lag4', 'Volume_lag5',
                     'roc_10_lag1', 'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
                     'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4', 'roc_21_lag5',
                     'macd_histogram_lag1', 'macd_histogram_lag2', 'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
                     'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5']]

In [22]:
X.shape

(71550, 30)

In [23]:
y = df[['label']]

In [24]:
y.shape

(71550, 1)

In [25]:
y

,label
Open time,
2018-01-02 14:00:00,1
2018-01-02 15:00:00,1
2018-01-02 16:00:00,1
2018-01-02 17:00:00,1
2018-01-02 18:00:00,1
...,...
2026-03-07 17:00:00,0
2026-03-07 18:00:00,0
2026-03-07 19:00:00,0


# Cross Validating the Model

In [26]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import time

# ================================================================
# STEP 1 — Define model parameters
# ================================================================

params = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'boosting_type':    'gbdt',
    'learning_rate':    0.05,
    'num_leaves':       31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'verbose':          -1
}

# ================================================================
# STEP 2 — Walk-Forward Cross Validation
# ================================================================

tscv = TimeSeriesSplit(n_splits=5)

all_predictions = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

    start = time.time()

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_test  = lgb.Dataset(X_test,  label=y_test, reference=lgb_train)

    model = lgb.train(
        params          = params,
        train_set       = lgb_train,
        num_boost_round = 300,
        valid_sets      = [lgb_test],
        callbacks       = [lgb.early_stopping(50), lgb.log_evaluation(50)]
    )

    fold_preds = model.predict(X_test)
    fold_df = y_test.copy()
    fold_df = fold_df.assign(pred_proba=fold_preds)
    fold_df['fold'] = fold + 1
    all_predictions.append(fold_df)

    elapsed = time.time() - start
    print(f'Fold {fold+1} done — {len(test_idx)} bars — {elapsed:.1f}s')

# ================================================================
# STEP 3 — Collect all predictions
# ================================================================

results = pd.concat(all_predictions)
print(results.head())
print(results.shape)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.691207
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.689923
Fold 1 done — 11925 bars — 0.3s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.693122
Early stopping, best iteration is:
[13]	valid_0's binary_logloss: 0.690732
Fold 2 done — 11925 bars — 0.3s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.694732
Early stopping, best iteration is:
[7]	valid_0's binary_logloss: 0.693366
Fold 3 done — 11925 bars — 0.3s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.690815
Early stopping, best iteration is:
[21]	valid_0's binary_logloss: 0.690116
Fold 4 done — 11925 bars — 0.5s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.693026
Early stopping, best iteration is:
[15]	valid_0's binary_logloss: 0.

In [27]:
results

,label,pred_proba,fold
Open time,,,
2019-05-17 17:00:00,1,0.491558,1
2019-05-17 18:00:00,1,0.493925,1
2019-05-17 19:00:00,1,0.488483,1
2019-05-17 20:00:00,1,0.515783,1
2019-05-17 21:00:00,1,0.526965,1
...,...,...,...
2026-03-07 17:00:00,0,0.513068,5
2026-03-07 18:00:00,0,0.553708,5
2026-03-07 19:00:00,0,0.552612,5


In [28]:
results.describe()

,label,pred_proba,fold
count,59625.000000,59625.000000,59625.000000
mean,0.511732,0.515055,3.000000
std,0.499867,0.031813,1.414225
min,0.000000,0.374510,1.000000
25%,0.000000,0.494616,2.000000
50%,1.000000,0.517656,3.000000
75%,1.000000,0.533519,4.000000
max,1.000000,0.685241,5.000000


# Training the Model

## Creating the Train / Test split according to user Input

In [63]:
# Hardcoding the date entered by user, to be set up dynamically in next versions
CUTOFF_DATE = '2025-01-01' 

# Creating the training set for X and y
X_train = X[X.index < CUTOFF_DATE]
y_train = y[y.index < CUTOFF_DATE]

# Creating the testting set for X and y
X_predict = X[X.index >= CUTOFF_DATE]
y_predict = y[y.index >= CUTOFF_DATE]  # kept for evaluation only

print(f'Training on {len(X_train)} bars up to {CUTOFF_DATE}')
print(f'Predicting on {len(X_predict)} bars from {CUTOFF_DATE} onwards')

Training on 61208 bars up to 2025-01-01
Predicting on 10342 bars from 2025-01-01 onwards


In [39]:
y_train.shape

(61208, 1)

In [40]:
X_train.shape

(61208, 30)

In [43]:
X_test.shape

(10342, 30)

In [44]:
y_test.shape

(10342, 1)

## Training the model with LightGBM

In [64]:
# Class imbalance correction
scale_pos_weight = float((y_train['label'] == 0).sum() / (y_train['label'] == 1).sum())

# Settting the parametrs needed for the model
params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'learning_rate':     0.05,
    'num_leaves':        31,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'scale_pos_weight':  scale_pos_weight,
    'seed':              42,
    'verbose':           -1
}

In [65]:
# Packaging the training data into LightGBM's required format
train_set_lgb = lgb.Dataset(X_train, label=y_train)

# Training the model
final_model= lgb.train(
    params= params,
    train_set= train_set_lgb,
    num_boost_round= 300,
    callbacks= [lgb.log_evaluation(period=-1)]
)

# Generate predictions on post-cutoff data
pred_proba_final = final_model.predict(X_predict)

In [66]:
pred_proba_final

array([0.40465365, 0.48252813, 0.40701168, ..., 0.57604269, 0.61581516,
       0.59047847], shape=(10342,))

## Building the signals DataFrame - this is the input to the execution filter

In [68]:
signals_df = df.loc[X_predict.index, ['Close', 'sma_50', 'sma_200', 'adx', 'atr_14']].copy()
signals_df['pred_proba'] = pred_proba_final
signals_df['true_label'] = y_predict.values

In [72]:
signals_df.head(3)

,Close,sma_50,sma_200,adx,atr_14,pred_proba,true_label
Open time,,,,,,,
2025-01-01 00:00:00,94401.14,93606.4300,95332.32805,17.514244,770.260715,0.404654,0
2025-01-01 01:00:00,93607.74,93603.8208,95333.54545,16.794643,774.524235,0.482528,1
2025-01-01 02:00:00,94098.91,93614.1198,95340.24725,16.126442,755.669647,0.407012,0


# Execution Filter + P&L Tracking

Purpose: Convert raw model probabilities into trade actions bar by bar

Why: Model output alone is not a trade — regime + confidence must confirm

MVP: Long only, bull regime only, confidence threshold filter

Entry metrics (stop distance, position size) are LOCKED at entry bar and never change during the trade — this prevents ATR drift from distorting stop levels and P&L calculations

Expandable: Add short trading in V2 by extending the bear regime block

Exit strategies used:
   1. Stop Loss     — price hits entry_price - trade_stop_dist (hard risk control)
   2. Regime Exit   — market structure no longer supports the trade
   3. Signal Exit   — model confidence drops below (1 - threshold)

In [74]:
# Setting the parameters of the execution strategy
CONFIDENCE_THRESHOLD = 0.55
INITIAL_CAPITAL = 1000.0
ATR_MULTIPLIER = 1.5
RISK_PCT = 0.01   # risk 1% of capital per trade
COST_PCT = 0.001  # 0.1% transaction cost per side

In [77]:
# Setting the loop that executes the trades

# State variables - maintained by the loop, not stored in Dataframe
position = 'flat'
entry_price = 0.0
trade_pos_size  = 0.0    # locked at entry, never changes mid-trade
trade_stop_dist = 0.0    # locked at entry, never changes mid-trade
capital = INITIAL_CAPITAL

# Storing the trade logs
trade_log = []

for timestamp, row in signals_df.iterrows():
    close   = row['Close']
    sma_50  = row['sma_50']
    sma_200 = row['sma_200']
    adx     = row['adx']
    atr     = row['atr_14']
    proba   = row['pred_proba']

    # Skip bars where indicators are not yet available (NaN warmup period)
    if pd.isna(atr) or pd.isna(adx) or pd.isna(sma_50) or pd.isna(sma_200):
        continue

    # Regime determination — checked every bar regardless of position
    # Bull requires price above both SMAs AND a strong trend (ADX > 25)
    # ADX < 25 = ranging/choppy market — signals unreliable
    bull = (close > sma_200) and (close > sma_50) and (adx > 25)

    # MANAGEMENT BLOCK — only runs when already in a trade (position == 'long')
    # Checks exit conditions in priority order:
    # Stop Loss → Regime Exit → Signal Exit

    if position == 'long':

        # EXIT 1: Stop Loss
        # Hard floor — exits immediately if price falls to locked stop level
        # stop_price is fixed at entry and never moves with ATR
        # This is the primary capital protection mechanism

        stop_price = entry_price - trade_stop_dist

        if close <= stop_price:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'stop_loss',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 2: Regime Exit
        # If market structure shifts out of bull regime, close the long
        # regardless of model signal — regime gate overrides model
        # MVP is long-only so no point holding in bear/sideways

        if not bull:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_regime',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 3: Signal Exit
        # Model confidence has flipped to strong SELL territory
        # Only exits if probability drops below (1 - threshold)
        # Probabilities near 0.5 do NOT trigger exit — only strong SELL signal does

        if proba <= (1-CONFIDENCE_THRESHOLD):
            pnl     = (close - entry_price) * trade_pos_size
            cost    = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_signal',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

    # ENTRY BLOCK - only runs when flat (position == 'flat')
    # All three conditions must be true simultaneously:
    #   1. Bull regime confirmed (regime gate)
    #   2. Model confidence above threshold (signal quality filter)
    # Entry metrics locked here — never recalculated during the trade

    else:
        if bull and proba >= CONFIDENCE_THRESHOLD:

            # Lock stop distance and position size at entry bar ATR
            # These values are frozen for the entire duration of this trade
            trade_stop_dist = ATR_MULTIPLIER * atr
            trade_pos_size = (capital * RISK_PCT) / trade_stop_dist

            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position = 'long'
            entry_price = close

            trade_log.append({
                'timestamp':  timestamp,
                'action':     'enter_long',
                'close':      close,
                'pnl':        0,
                'capital':    capital
            })

## Performance Summary

In [83]:
# Convert trade log to DataFrame for analysis
trade_df = pd.DataFrame(trade_log)

# Separate entries from exits
entries = trade_df[trade_df['action'] == 'enter_long']
exits   = trade_df[trade_df['action'] != 'enter_long']

# Win rate — trades where PnL > 0
winning_trades = exits[exits['pnl'] > 0]
losing_trades  = exits[exits['pnl'] < 0]

total_closed = len(exits)
win_rate     = len(winning_trades) / total_closed * 100 if total_closed > 0 else 0
loss_rate    = len(losing_trades)  / total_closed * 100 if total_closed > 0 else 0

print(f'\n=== BACKTEST RESULTS ===')
print(f'Initial capital:  ${INITIAL_CAPITAL:,.2f}')
print(f'Final capital:    ${capital:,.2f}')
print(f'Total return:     {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):.2f}%')
print(f'\n=== TRADE STATISTICS ===')
print(f'Total trades:     {len(entries)}')
print(f'Winning trades:   {len(winning_trades)}')
print(f'Losing trades:    {len(losing_trades)}')
print(f'Win rate:         {win_rate:.1f}%')
print(f'Loss rate:        {loss_rate:.1f}%')
print(f'\n=== ACTION BREAKDOWN ===')
print(trade_df['action'].value_counts())


=== BACKTEST RESULTS ===
Initial capital:  $1,000.00
Final capital:    $777.00
Total return:     -22.30%

=== TRADE STATISTICS ===
Total trades:     123
Winning trades:   65
Losing trades:    58
Win rate:         52.8%
Loss rate:        47.2%

=== ACTION BREAKDOWN ===
action
enter_long           123
close_long_signal     72
close_long_regime     31
stop_loss             20
Name: count, dtype: int64


In [82]:
# Convert trade log to DataFrame for analysis
trade_df = pd.DataFrame(trade_log)

print(f'\n=== BACKTEST RESULTS ===')
print(f'Initial capital: ${INITIAL_CAPITAL:,.2f}')
print(f'Final capital:   ${capital:,.2f}')
print(f'Total return:    {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):.2f}%')
print(f'Total trades:    {len(trade_df)}')
if len(trade_df) > 0:
    print(f'\nAction breakdown:')
    print(trade_df['action'].value_counts())
    print(f'\nSample trades:')
    print(trade_df.head(10))


=== BACKTEST RESULTS ===
Initial capital: $1,000.00
Final capital:   $777.00
Total return:    -22.30%
Total trades:    246

Action breakdown:
action
enter_long           123
close_long_signal     72
close_long_regime     31
stop_loss             20
Name: count, dtype: int64

Sample trades:
            timestamp             action      close        pnl     capital
0 2025-01-02 22:00:00         enter_long   96911.04   0.000000  999.056101
1 2025-01-03 08:00:00  close_long_regime   96129.97  -7.607506  990.512303
2 2025-01-04 03:00:00         enter_long   98200.00   0.000000  989.087341
3 2025-01-04 07:00:00  close_long_signal   98271.88   1.043037  988.704374
4 2025-01-07 04:00:00         enter_long  101667.02   0.000000  987.536987
5 2025-01-07 12:00:00          stop_loss  100750.89 -10.519420  975.860700
6 2025-01-07 13:00:00         enter_long  100636.80   0.000000  974.525373
7 2025-01-07 14:00:00  close_long_regime   99979.69  -8.719043  964.479722
8 2025-01-16 04:00:00         ent

In [79]:
# Buy and Hold benchmark
initial_capital = 1000.0
buy_date        = '2025-01-01'

# Price at buy date — first available bar on or after buy date
buy_price  = df.loc[buy_date:, 'Close'].iloc[0]

# Latest available price in dataset
final_price = df['Close'].iloc[-1]

# Calculate return
btc_units    = initial_capital / buy_price
final_value  = btc_units * final_price
total_return = ((final_value - initial_capital) / initial_capital) * 100

print(f'Buy price  ({buy_date}): ${buy_price:,.2f}')
print(f'Final price:              ${final_price:,.2f}')
print(f'BTC units held:           {btc_units:.6f}')
print(f'Initial capital:          ${initial_capital:,.2f}')
print(f'Final value:              ${final_value:,.2f}')
print(f'Total return:             {total_return:.2f}%')
print(f'\nStrategy return:          -22.30%')
print(f'Buy & Hold return:        {total_return:.2f}%')

Buy price  (2025-01-01): $94,401.14
Final price:              $67,452.98
BTC units held:           0.010593
Initial capital:          $1,000.00
Final value:              $714.54
Total return:             -28.55%

Strategy return:          -22.30%
Buy & Hold return:        -28.55%


In [80]:
trade_df

,timestamp,action,close,pnl,capital
0,2025-01-02 22:00:00,enter_long,96911.04,0.000000,999.056101
1,2025-01-03 08:00:00,close_long_regime,96129.97,-7.607506,990.512303
2,2025-01-04 03:00:00,enter_long,98200.00,0.000000,989.087341
3,2025-01-04 07:00:00,close_long_signal,98271.88,1.043037,988.704374
4,2025-01-07 04:00:00,enter_long,101667.02,0.000000,987.536987
...,...,...,...,...,...
241,2026-03-04 23:00:00,close_long_signal,72666.77,-0.240227,775.946271
242,2026-03-05 07:00:00,enter_long,72055.50,0.000000,775.447951
243,2026-03-05 10:00:00,close_long_signal,73388.92,9.221627,784.162038
244,2026-03-05 15:00:00,enter_long,71652.77,0.000000,783.691258


In [84]:
print(f'Avg winning PnL: ${winning_trades["pnl"].mean():,.2f}')
print(f'Avg losing PnL:  ${losing_trades["pnl"].mean():,.2f}')

Avg winning PnL: $6.55
Avg losing PnL:  $-6.86


### Calculating Sharpe Ratios

In [86]:
# Calculate bar by bar returns from capital column
trade_df_sorted = trade_df.sort_values('timestamp')
trade_df_sorted['capital_return'] = trade_df_sorted['capital'].pct_change()

# Sharpe Ratio — annualised for hourly bars
# 8760 = hours in a year
mean_return   = trade_df_sorted['capital_return'].mean()
std_return    = trade_df_sorted['capital_return'].std()
sharpe        = (mean_return / std_return) * np.sqrt(8760)

# Maximum Drawdown
trade_df_sorted['cummax'] = trade_df_sorted['capital'].cummax()
trade_df_sorted['drawdown'] = (trade_df_sorted['capital'] - trade_df_sorted['cummax']) / trade_df_sorted['cummax']
max_drawdown  = trade_df_sorted['drawdown'].min() * 100

# Profit Factor
gross_profit  = winning_trades['pnl'].sum()
gross_loss    = losing_trades['pnl'].abs().sum()
profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0

# Annualised Return
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

print(f'\n=== RISK METRICS ===')
print(f'Sharpe Ratio:      {sharpe:.2f}')
print(f'Max Drawdown:      {max_drawdown:.2f}%')
print(f'Profit Factor:     {profit_factor:.2f}')
print(f'Annualised Return: {annualised_return:.2f}%')


=== RISK METRICS ===
Sharpe Ratio:      -13.37
Max Drawdown:      -23.49%
Profit Factor:     1.07
Annualised Return: -19.44%


# Iteration with different trading execuion Strategy

In [95]:
# Setting the parameters of the execution strategy
CONFIDENCE_THRESHOLD = 0.60
INITIAL_CAPITAL = 1000.0
ATR_MULTIPLIER = 1.5
RISK_PCT = 0.01   # risk 1% of capital per trade
COST_PCT = 0.001  # 0.1% transaction cost per side

# State variables - maintained by the loop, not stored in Dataframe
position = 'flat'
entry_price = 0.0
trade_pos_size  = 0.0    # locked at entry, never changes mid-trade
trade_stop_dist = 0.0    # locked at entry, never changes mid-trade
capital = INITIAL_CAPITAL

# Storing the trade logs
trade_log = []

for timestamp, row in signals_df.iterrows():
    close   = row['Close']
    sma_50  = row['sma_50']
    sma_200 = row['sma_200']
    adx     = row['adx']
    atr     = row['atr_14']
    proba   = row['pred_proba']

    # Skip bars where indicators are not yet available (NaN warmup period)
    if pd.isna(atr) or pd.isna(adx) or pd.isna(sma_50) or pd.isna(sma_200):
        continue

    # Regime determination — checked every bar regardless of position
    # Bull requires price above both SMAs AND a strong trend (ADX > 25)
    # ADX < 25 = ranging/choppy market — signals unreliable
    bull = (close > sma_200) and (close > sma_50) and (adx > 25)

    # MANAGEMENT BLOCK — only runs when already in a trade (position == 'long')
    # Checks exit conditions in priority order:
    # Stop Loss → Regime Exit → Signal Exit

    if position == 'long':

        # EXIT 1: Stop Loss
        # Hard floor — exits immediately if price falls to locked stop level
        # stop_price is fixed at entry and never moves with ATR
        # This is the primary capital protection mechanism

        stop_price = entry_price - trade_stop_dist

        if close <= stop_price:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'stop_loss',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 2: Regime Exit
        # If market structure shifts out of bull regime, close the long
        # regardless of model signal — regime gate overrides model
        # MVP is long-only so no point holding in bear/sideways

        if not bull:
            pnl = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_regime',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            # Reset all trade state — trade_stop_dist also reset to avoid stale values
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

        # EXIT 3: Signal Exit
        # Model confidence has flipped to strong SELL territory
        # Only exits if probability drops below (1 - threshold)
        # Probabilities near 0.5 do NOT trigger exit — only strong SELL signal does

        if proba <= (1-CONFIDENCE_THRESHOLD):
            pnl     = (close - entry_price) * trade_pos_size
            cost    = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp,
                'action':    'close_long_signal',
                'close':     close,
                'pnl':       pnl,
                'capital':   capital
            })
            position, entry_price, trade_pos_size, trade_stop_dist = 'flat', 0.0, 0.0, 0.0
            continue

    # ENTRY BLOCK - only runs when flat (position == 'flat')
    # All three conditions must be true simultaneously:
    #   1. Bull regime confirmed (regime gate)
    #   2. Model confidence above threshold (signal quality filter)
    # Entry metrics locked here — never recalculated during the trade

    else:
        if bull and proba >= CONFIDENCE_THRESHOLD:

            # Lock stop distance and position size at entry bar ATR
            # These values are frozen for the entire duration of this trade
            trade_stop_dist = ATR_MULTIPLIER * atr
            trade_pos_size = (capital * RISK_PCT) / trade_stop_dist

            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position = 'long'
            entry_price = close

            trade_log.append({
                'timestamp':  timestamp,
                'action':     'enter_long',
                'close':      close,
                'pnl':        0,
                'capital':    capital
            })

In [96]:
import numpy as np

# ================================================================
# FULL PERFORMANCE EVALUATION
# ================================================================

# --- Setup ---
trade_df       = pd.DataFrame(trade_log)
entries        = trade_df[trade_df['action'] == 'enter_long']
exits          = trade_df[trade_df['action'] != 'enter_long']
winning_trades = exits[exits['pnl'] > 0]
losing_trades  = exits[exits['pnl'] < 0]
total_closed   = len(exits)
win_rate       = len(winning_trades) / total_closed * 100 if total_closed > 0 else 0
loss_rate      = len(losing_trades)  / total_closed * 100 if total_closed > 0 else 0

# --- Risk Metrics ---
trade_df_sorted = trade_df.sort_values('timestamp')
trade_df_sorted['capital_return'] = trade_df_sorted['capital'].pct_change()
mean_return      = trade_df_sorted['capital_return'].mean()
std_return       = trade_df_sorted['capital_return'].std()
sharpe           = (mean_return / std_return) * np.sqrt(8760)
trade_df_sorted['cummax']   = trade_df_sorted['capital'].cummax()
trade_df_sorted['drawdown'] = (trade_df_sorted['capital'] - trade_df_sorted['cummax']) / trade_df_sorted['cummax']
max_drawdown     = trade_df_sorted['drawdown'].min() * 100
gross_profit     = winning_trades['pnl'].sum()
gross_loss       = losing_trades['pnl'].abs().sum()
profit_factor    = gross_profit / gross_loss if gross_loss > 0 else 0
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

# --- Buy and Hold Benchmark ---
buy_date    = CUTOFF_DATE
buy_price   = df.loc[buy_date:, 'Close'].iloc[0]
final_price = df['Close'].iloc[-1]
btc_units   = INITIAL_CAPITAL / buy_price
bnh_value   = btc_units * final_price
bnh_return  = ((bnh_value - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100

# --- Transaction Costs ---
implied_costs = trade_df['pnl'].sum() - (capital - INITIAL_CAPITAL)

# --- Print All Results ---
print(f'╔══════════════════════════════════════╗')
print(f'║         BACKTEST RESULTS             ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ CAPITAL                              ║')
print(f'║  Initial:         ${INITIAL_CAPITAL:>10,.2f}       ║')
print(f'║  Final:           ${capital:>10,.2f}       ║')
print(f'║  Total Return:    {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%       ║')
print(f'║  Annualised:      {annualised_return:>9.2f}%       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ TRADE STATISTICS                     ║')
print(f'║  Total Trades:    {len(entries):>10}       ║')
print(f'║  Winning Trades:  {len(winning_trades):>10}       ║')
print(f'║  Losing Trades:   {len(losing_trades):>10}       ║')
print(f'║  Win Rate:        {win_rate:>9.1f}%       ║')
print(f'║  Loss Rate:       {loss_rate:>9.1f}%       ║')
print(f'║  Avg Win PnL:     ${winning_trades["pnl"].mean():>10.2f}       ║')
print(f'║  Avg Loss PnL:    ${losing_trades["pnl"].mean():>10.2f}       ║')
print(f'║  Implied Costs:   ${implied_costs:>10.2f}       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ RISK METRICS                         ║')
print(f'║  Sharpe Ratio:    {sharpe:>10.2f}       ║')
print(f'║  Max Drawdown:    {max_drawdown:>9.2f}%       ║')
print(f'║  Profit Factor:   {profit_factor:>10.2f}       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ BUY & HOLD BENCHMARK                 ║')
print(f'║  Buy Price:       ${buy_price:>10,.2f}       ║')
print(f'║  Final Price:     ${final_price:>10,.2f}       ║')
print(f'║  Final Value:     ${bnh_value:>10,.2f}       ║')
print(f'║  B&H Return:      {bnh_return:>9.2f}%       ║')
print(f'║  Strategy Return: {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ ACTION BREAKDOWN                     ║')
for action, count in trade_df['action'].value_counts().items():
    print(f'║  {action:<20} {count:>6}          ║')
print(f'╚══════════════════════════════════════╝')

╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║ CAPITAL                              ║
║  Initial:         $  1,000.00       ║
║  Final:           $    859.82       ║
║  Total Return:       -14.02%       ║
║  Annualised:         -12.14%       ║
╠══════════════════════════════════════╣
║ TRADE STATISTICS                     ║
║  Total Trades:            65       ║
║  Winning Trades:          33       ║
║  Losing Trades:           32       ║
║  Win Rate:             50.8%       ║
║  Loss Rate:            49.2%       ║
║  Avg Win PnL:     $      8.14       ║
║  Avg Loss PnL:    $     -8.70       ║
║  Implied Costs:   $    130.47       ║
╠══════════════════════════════════════╣
║ RISK METRICS                         ║
║  Sharpe Ratio:        -13.32       ║
║  Max Drawdown:       -15.09%       ║
║  Profit Factor:         0.97       ║
╠══════════════════════════════════════╣
║ BUY & HOLD BENCHMARK                 ║
